# Pretrained ResNet: следующий fine-tuning

По умолчанию — Average+Max pooling, инициализация из уже обученного best ResNet101. Это новый эксперимент, не повтор старого обучения. Выберите конфиг и уникальный EXP_ID, запустите сверху вниз. Обоснование: `docs/global_local_review.md`.

Нужен собственный `artifacts/models/resnet101_partial_domain_shift_stronger_001/best.pt`. На Kaggle перенесите его вместе с проектом или измените training.initialize_from. ImageNet повторно не скачивается. 12 эпох, первый head warmup=1; partial layer3/layer4, frozen BN. Остальные конфиги остаются доступны, их warmup задаётся отдельно.

Общий pipeline печатает метрики, рисует графики и сохраняет журнал/артефакты. DEVICE=auto выбирает CUDA при наличии, иначе CPU. Этот fine-tuning рекомендуется выполнить на Kaggle GPU. Test inference выключен.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "experiment.py").exists():
    raise RuntimeError("Откройте ноутбук из корня проекта или notebooks/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.experiment import load_config, run_experiment
from src.utils import get_device


In [ ]:
# Новый кандидат: trained ResNet101 -> Average+Max, 12 эпох fine-tuning.
CONFIG = "configs/resnet101_partial_avgmax_domain_shift.json"
EXP_ID = "resnet101_partial_avgmax_domain_shift_001"  # Уникальный ID
DEVICE = "auto"

config = load_config(CONFIG, project_root=PROJECT_ROOT)
config["training"]["device"] = DEVICE
# Если batch 32 не помещается в GPU, уменьшите batch_size, сохраняя image_size=256.
# config["data"]["batch_size"] = 16
print(f"Конфигурация: {CONFIG}; устройство: {get_device(DEVICE)}")

# Если источник находится в Kaggle Input, задайте абсолютный путь:
# config["training"]["initialize_from"] = "/kaggle/input/your-checkpoint/best.pt"


In [ ]:
result = run_experiment(config, exp_id=EXP_ID, project_root=PROJECT_ROOT)


## После обучения

Сохраните artifacts/models, artifacts/logs, artifacts/predictions и новую строку experiments.csv. Новый pipeline сохраняет также val_probabilities.npz: IDs, порядок классов, вероятности и true_labels.
Для сравнения TTA/ансамбля откройте 06_tta_ensembles.ipynb и добавьте новый best.pt в CHECKPOINTS, задав новый INFERENCE_RUN_ID. Test не используется при обучении и выборе.
